# Fase 1: Construção do Grafo

Este notebook implementa a transformação do dataset bruto de conexões em matrizes esparsas eficientes utilizando o módulo `src.graph.builder`. Esta matriz é a estrutura base para a análise topológica e modelos de redes neurais futuras.

In [ ]:
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import time
import sys
import os

# Adiciona o diretório src ao path para podermos importar nossos módulos locais
sys.path.append(os.path.abspath('..'))
from src.graph.builder import ConnectomeBuilder

## 1. Carregamento dos Dados

In [ ]:
# Carregando o dataset local (já validado na Fase 0)
DATA_PATH = '../data/raw/connections_princeton.csv.gz'

print(f"Carregando {DATA_PATH}...")
start = time.time()
df = pd.read_csv(DATA_PATH)
print(f"Carregamento concluído em {time.time() - start:.2f}s")
print(f"Total de linhas no dataset bruto: {len(df):,}")

## 2. Construção da Matriz Esparsa
Neste passo agrupamos múltiplas conexões entre os mesmos neurônios (ex: conexões em diferentes regiões) e montamos a Matriz CSR.

In [ ]:
builder = ConnectomeBuilder(df)

# Construindo a matriz utilizando syn_count como peso da aresta
adj_matrix, mapper = builder.build_sparse_matrix(weight_col='syn_count')

## 3. Validação da Matriz vs Tabela
Vamos confirmar se a matriz consumiu menos memória e se os totais biológicos batem.

In [ ]:
print(f"Tamanho da Matriz: {adj_matrix.shape}")
print(f"Número total de neurônios únicos mapeados: {mapper.num_nodes:,}")
print(f"Número total de pares de conexões únicas (arestas do grafo): {adj_matrix.nnz:,}")

total_synapses = adj_matrix.sum()
print(f"Total de sinapses (soma dos pesos): {total_synapses:,}")

# Comparativo de Memória
df_memory_mb = df.memory_usage(deep=True).sum() / (1024**2)
matrix_memory_mb = (adj_matrix.data.nbytes + adj_matrix.indptr.nbytes + adj_matrix.indices.nbytes) / (1024**2)

print(f"\nMemória gasta pelo DataFrame original: {df_memory_mb:.2f} MB")
print(f"Memória gasta pela Matriz Esparsa (CSR): {matrix_memory_mb:.2f} MB")
print(f"Economia de espaço na memória RAM: {df_memory_mb / matrix_memory_mb:.1f}x")

## 4. Teste de Acesso (Mapeador)
Vamos pegar o mesmo neurônio que foi testado na Fase 0 e verificar como buscar ele na matriz.

In [ ]:
# Neurônio de teste da Fase 0
test_neuron_root_id = 720575940625363947

# Converte para o índice matemático contínuo
matrix_idx = mapper.get_idx(test_neuron_root_id)

print(f"ID Biológico (root_id): {test_neuron_root_id}")
print(f"Índice na Matriz (0 a N): {matrix_idx}")

# Buscando as conexões de saída (Out-Degree ponderado)
if matrix_idx is not None:
    outgoing_row = adj_matrix[matrix_idx, :]
    total_outgoing_partners = outgoing_row.nnz
    total_outgoing_synapses = outgoing_row.sum()
    print(f"\nParceiros de saída únicos: {total_outgoing_partners}")
    print(f"Total de sinapses de saída: {total_outgoing_synapses}")
else:
    print("Neurônio não encontrado no dataset.")

## 5. Extração de Subgrafo (NetworkX)
Como orientado no `AGENTS.md`, o NetworkX não deve ser usado para a rede toda. Vamos usá-lo apenas para visualizar ou processar uma vizinhança muito pequena, como todos os parceiros imediatos de um neurônio específico.

In [ ]:
# Cria um subgrafo focado nos parceiros do nosso neurônio de teste
if matrix_idx is not None:
    # Obtém os índices dos parceiros (quem ele envia sinal)
    partners_idx = outgoing_row.indices
    
    # Criamos uma submatriz (Nós + Vizinhos) usando os vizinhos
    subgraph_nodes = [matrix_idx] + list(partners_idx)
    sub_matrix = adj_matrix[subgraph_nodes, :][:, subgraph_nodes]
    
    # Converte essa sub-matriz pequena para NetworkX
    G = nx.from_scipy_sparse_array(sub_matrix, create_using=nx.DiGraph)
    
    print(f"Subgrafo extraído e convertido para NetworkX.")
    print(f"Nós no subgrafo: {G.number_of_nodes()}")
    print(f"Arestas no subgrafo: {G.number_of_edges()}")
    
    # Recuperando o root_id de algum vizinho aleatório
    first_neighbor_idx = subgraph_nodes[1]
    print(f"\nPrimeiro parceiro biológico (root_id): {mapper.get_root_id(first_neighbor_idx)}")